# Incremental generation prototype
- 3개 생성 후 또 3개 생성.. until 원래 max_tokens까지 생성될 때 까지 다 생성되면 stop 
- 모든 길이를 비교 

## 실험하기 위한 세팅 (데이터 불러오기 변수 설정하기)

In [1]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple
from copy import deepcopy

from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0, CustomDataset, repeat_interleave_unravel, analyze_span_lengths_and_count
import new_module.losses as lossbuilder

config = {'task': 'toxicity',
        'device': 'cuda',
        'losses': ['gpt2', 'classification_no_prefix_logprobloss'],
        'cache_dir': '/data/hyeryung/hf_cache',
        'model_paths': ['Qwen/Qwen2.5-7B',
                        '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'],
        'model_types': ['AutoModelForCausalLM', 'AutoModelForSequenceClassification'],
        'build_loss_dict': {'AR_top_k': 0,
                            'AR_top_p': 0.96,
                            'loss_type': 'xentropy',
                            'coeff_steps': 200,
                            'coeff_pattern': 'constant',
                            'AR_temperature': 1,
                            'length_normalize': False, #True, # default False
                            'max_output_length': 20},
        'tokenizer_paths': ['Qwen/Qwen2.5-7B',
                        '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'],
        'max_tokens_per_span': 3,
        'consider_prompt_for_cand_gen': True,
        'k_per_location': 10,
        'loss_weights': [0.1, 1.0],
        'target_label_ids': [0, 0],
        'beam_size': 5,
        'min_epsilons': [0.95],
        'selection_criteria': 'allsat_primary'
    }

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**config["build_loss_dict"])
build_loss_args.task = config["task"]

mlm = AutoModelForMaskedLM.from_pretrained('roberta-large').to(config['device'])
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-large')

## load tokenizer, models, define losses
name2tokenizer = {}
name2model = {}
name2config = {}
loss2tokenizer = {}
embed_luts = []

for i, model_path in enumerate(config["model_paths"]):
    if (
        model_path not in name2model
    ):  # making sure we are not loading the model twice in case some constraints use the same model.
        try:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=True,
            )
        except:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=False,
            )

        name2config[model_path] = AutoConfig.from_pretrained(
            model_path, cache_dir=config["cache_dir"]
        )

        if config["model_types"][i] == "RobertaCustomForSequenceClassification":
            pass
        else:
            name2model[model_path] = lossbuilder.ModelWrapper(
                getattr(transformers, config["model_types"][i]).from_pretrained(
                    model_path,
                    config=name2config[model_path],
                    cache_dir=config["cache_dir"],
                )
            )
        name2model[model_path].eval()
        name2model[model_path].to(config['device'])

lossfns = []
for i, loss in enumerate(config["losses"]):
    lossfns.append(
        lossbuilder.build_loss(
            loss,
            name2model[config["model_paths"][i]],
            name2tokenizer[config["tokenizer_paths"][i]],
            build_loss_args,
        )
    )
    lossfns[i].tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})
    loss2tokenizer[loss] = lossfns[i].tokenizer

special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)

Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/config.json HTTP/11" 200 0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/generation_config.json HTTP/11" 200 0


In [3]:

source_text = "That means audiences will again see Andrew W.K. perform classic cuts like “Party Hard,"
original_sent = "“I Get Wet,” and “Ready to Die” that always get the crowd hyped up and dancing like there's no tomorrow. Andrew W.K.'s high-energy performances and infectious enthusiasm are sure to make the concert an unforgettable experience for everyone in attendance. His music has a way of bringing people together and creating a sense of pure, unadulterated joy that is hard to find elsewhere. Fans can expect a night filled with non-stop partying and a celebration of life like no other."
test_sent = "<mask><mask><mask><mask><mask><mask><mask><mask> and<mask><mask><mask> to<mask><mask><mask> that always get the crowd hyped up and dancing like there's no tomorrow. Andrew W.K.'s high-energy performances and infectious enthusiasm are sure to make the<mask> an unforgettable experience for everyone in attendance. His music has a way of bringing people together and creating a sense of pure, unadulterated joy that is hard to find elsewhere. Fans can expect a night filled with non-stop<mask> and a celebration of life like no other."
test_sent_curr_span = "<mask><mask><mask><mask><mask><mask><mask><mask> and<mask><mask><mask> to<mask><mask><mask> that always get the crowd hyped up and dancing like there's no tomorrow. Andrew W.K.'s high-energy performances and infectious enthusiasm are sure to make the<mask> an unforgettable experience for everyone in attendance. His music has a way of bringing people together and creating a sense of pure, unadulterated joy that is hard to find elsewhere. Fans can expect a night filled with non-stop<mask> and a celebration of life like no other."
mask_info_dict, span_lengths = analyze_span_lengths_and_count(test_sent_curr_span)

In [5]:
# merge masks
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent_curr_span)

# Max number of mask tokens to replace each span
# max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]
max_mask_cnt_per_span = [config['max_tokens_per_span'] for x in span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])


In [6]:
# for i in range(len(mask_spans)):
i = 0 ### for now, we only consider the first span
curr_queue_size = len(queue)

# candidate generation
curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
## Tokenize & conduct MLM inference
inputs = mlm_tokenizer(
    curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True, add_special_tokens=False
)
inputs = inputs.to(config['device']) 
masked_sequence=inputs['input_ids']

# if config['consider_prompt_for_cand_gen']: # default True
    
prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
# print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")

input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])

# with torch.no_grad():
#     logits = mlm(**inputs).logits
with torch.no_grad():
    logits = mlm(input_ids = input_tokens, 
                attention_mask = attention_masks).logits

# Choose top k among non-special tokens
# print(f"-- shape of logits before removing source text part: {logits.shape}")
logits = logits[:, prompt_enc.input_ids.shape[1]:]

## Choose top k among non-special tokens
logits[:, :, special_token_ids] = -float("inf")

indices_in_mlm_tokens = (
    inputs.input_ids == mlm_tokenizer.mask_token_id
).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
# print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")


In [7]:

## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
# print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
# print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")

## Get top k tokens for the j masks
predicted_token_ids = torch.topk(
    logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
    k=config['k_per_location'],
    dim=-1,
).indices            

In [8]:
for x in predicted_token_ids:
    print(mlm_tokenizer.batch_decode(x))

['�', ' and', ' the', ' �', ' Baby', ' Get', ' I', ' The', ',', ' but']
['�', '�', '�', '�', ' �', ',', ' more', ' on', '�', ' the']
[' live', ' —', ' Live', ',', ' with', ' energetic', ' –', '�', ' onstage', ' ,']


In [9]:
masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[max_mask_cnt_per_span[i]*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)

In [10]:
# 인자 설정
source_text=source_text
masked_sequence=masked_sequence
indices_in_mlm_tokens=(indices_in_mlm_tokens_0, indices_in_mlm_tokens_1)
predicted_token_ids=predicted_token_ids
mlm_tokenizer=mlm_tokenizer
lossfns=lossfns
config=config
return_all_hypotheses=True
primary_loss_only=False,
batch_size=16

In [11]:
# 여기서부터 함수 내부 동작
# 변수 초기화
final_hypotheses = [[] for i in range(len(masked_sequence))]
final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))
loss_weights = config['loss_weights']

In [12]:
# variable replacement 케이스 처리
# for curr_edit_index in edit_indices: ## 이게 사실상 "j=1…mask 최대 개수 까지" 와 같다.
curr_edit_index = 12        
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
print(f"batch_ids_to_edit: {batch_ids_to_edit}") # [0]
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
print(f"tmp_hypotheses: {tmp_hypotheses}")
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
new_func_candidates = new_func_candidates.to(config['device'])
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"len(tmp_hypotheses): {len(tmp_hypotheses)}")
print(f"tmp_hypotheses: {tmp_hypotheses}")

logging_loss = torch.zeros((len(tmp_hypotheses), len(lossfns))).to(config['device'])
curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
tmp_hypotheses_dec = mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True)
data_loader = DataLoader(CustomDataset(tmp_hypotheses_dec),batch_size=batch_size)
for lossid, lossname in enumerate(config["losses"]):
    lossvalues=[]
    with torch.no_grad():
        for batch in data_loader:
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, batch,
                label_id=config['target_label_ids'][lossid],
            )
            lossvalues.append(lossvalue)
            torch.cuda.empty_cache()
    lossvalue = torch.cat(lossvalues,dim=0)
    curr_loss += loss_weights[lossid] * lossvalue
    logging_loss[:,lossid] = lossvalue


batch_ids_to_edit: []
num_initial_tmp_hypotheses: []


RuntimeError: torch.cat(): expected a non-empty list of Tensors

In [ ]:
logging_loss

tensor([[   37.981,     0.023],
        [   43.391,     0.050],
        [   41.253,     0.031],
        [   43.063,     0.025],
        [   44.617,     0.031],
        [   43.653,     0.044],
        [   42.753,     0.023],
        [   38.871,     0.016],
        [   42.105,     0.026],
        [   43.222,     0.022]], device='cuda:0')

In [ ]:
curr_loss

tensor([3.821, 4.389, 4.156, 4.331, 4.493, 4.409, 4.298, 3.903, 4.236, 4.344],
       device='cuda:0')

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,0].argsort().resize(10,1)].squeeze(),skip_special_tokens=True)

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/torch/_tensor.py:889: UserWarning: non-inplace resize is deprecated
  warnings.warn("non-inplace resize is deprecated")


[' before my wife found out I was a man, I was',
 ' before my wife found out I was a man, I had',
 ' before my wife found out I was a man, I thought',
 ' before my wife found out I was a man, I did',
 ' before my wife found out I was a man, I knew',
 ' before my wife found out I was a man, I believed',
 ' before my wife found out I was a man, I felt',
 ' before my wife found out I was a man, I wasn',
 ' before my wife found out I was a man, I never',
 ' before my wife found out I was a man, I wondered']

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,1].argsort().resize(10,1)].squeeze(),skip_special_tokens=True)

[' before my wife found out I was a man, I had',
 ' before my wife found out I was a man, I felt',
 ' before my wife found out I was a man, I was',
 ' before my wife found out I was a man, I knew',
 ' before my wife found out I was a man, I believed',
 ' before my wife found out I was a man, I did',
 ' before my wife found out I was a man, I wondered',
 ' before my wife found out I was a man, I thought',
 ' before my wife found out I was a man, I never',
 ' before my wife found out I was a man, I wasn']

In [ ]:
torch.cat([logging_loss[:,0].argsort().resize(10,1),logging_loss[:,1].argsort().resize(10,1),curr_loss.argsort().resize(10,1)],dim=-1)

tensor([[0, 7, 0],
        [7, 9, 7],
        [2, 0, 2],
        [8, 6, 8],
        [6, 3, 6],
        [3, 8, 3],
        [9, 4, 9],
        [1, 2, 1],
        [5, 5, 5],
        [4, 1, 4]], device='cuda:0')

In [ ]:
        
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print(f"final_hypotheses: {final_hypotheses}")
print(f"final_hypotheses_losses: {final_hypotheses_losses}")

curr_loss: tensor([3.821, 4.389, 4.156, 4.331, 4.493, 4.409, 4.298, 3.903, 4.236, 4.344],
       device='cuda:0')
curr_loss: (tensor([3.821, 4.389, 4.156, 4.331, 4.493, 4.409, 4.298, 3.903, 4.236, 4.344],
       device='cuda:0'),)
top_beams: [tensor([0, 7, 2, 8, 6], device='cuda:0')]
tmp_hypotheses: (tensor([[    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,    21],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,   938],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,   802],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,  2047],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38, 12267],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,   393],
        [    0,   137,   127,  1141,  

In [ ]:
mlm_tokenizer.batch_decode(final_hypotheses[0])

['<s> before my wife found out I was a man, I was',
 '<s> before my wife found out I was a man, I had',
 '<s> before my wife found out I was a man, I thought',
 '<s> before my wife found out I was a man, I did',
 '<s> before my wife found out I was a man, I knew']

In [ ]:
# 다음 step 
curr_edit_index = 13        
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
print(f"batch_ids_to_edit: {batch_ids_to_edit}") # [0]
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
new_func_candidates = new_func_candidates.to(config['device'])
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"len(tmp_hypotheses): {len(tmp_hypotheses)}")
print(f"tmp_hypotheses: {tmp_hypotheses}")

logging_loss = torch.zeros((len(tmp_hypotheses), len(lossfns))).to(config['device'])
curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
tmp_hypotheses_dec = mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True)
data_loader = DataLoader(CustomDataset(tmp_hypotheses_dec),batch_size=batch_size)
for lossid, lossname in enumerate(config["losses"]):
    lossvalues=[]
    with torch.no_grad():
        for batch in data_loader:
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, batch,
                label_id=config['target_label_ids'][lossid],
            )
            lossvalues.append(lossvalue)
            torch.cuda.empty_cache()
    lossvalue = torch.cat(lossvalues,dim=0)
    curr_loss += loss_weights[lossid] * lossvalue
    logging_loss[:,lossid] = lossvalue


batch_ids_to_edit: [0]
num_initial_tmp_hypotheses: [50]
len(tmp_hypotheses): 50
tmp_hypotheses: tensor([[    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,    21,    75],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,    56,    75],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,   802,    75],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,   222,    75],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,  1467,    75],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,    21, 12267],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
             6,    38,    56, 12267],
        [    0,   137,   127,  1141,   303,    66,    38,    21,    10,   313,
      

In [ ]:
logging_loss

tensor([[   52.087,     0.051],
        [   53.105,     0.027],
        [   55.366,     0.034],
        [   50.584,     0.031],
        [   58.012,     0.028],
        [   50.551,     0.028],
        [   47.404,     0.019],
        [   55.784,     0.032],
        [   53.927,     0.024],
        [   56.527,     0.028],
        [   47.287,     0.024],
        [   47.707,     0.022],
        [   43.722,     0.022],
        [   49.022,     0.026],
        [   45.844,     0.019],
        [   49.465,     0.040],
        [   47.888,     0.018],
        [   47.823,     0.025],
        [   46.752,     0.030],
        [   49.137,     0.018],
        [   44.497,     0.062],
        [   46.543,     0.021],
        [   50.033,     0.035],
        [   44.077,     0.027],
        [   50.613,     0.032],
        [   42.195,     0.030],
        [   42.908,     0.019],
        [   45.195,     0.025],
        [   45.328,     0.026],
        [   46.479,     0.018],
        [   47.552,     0.034],
        

In [ ]:
curr_loss

tensor([5.260, 5.338, 5.570, 5.089, 5.829, 5.084, 4.759, 5.611, 5.417, 5.680,
        4.753, 4.792, 4.394, 4.928, 4.603, 4.987, 4.807, 4.808, 4.705, 4.931,
        4.512, 4.675, 5.038, 4.435, 5.093, 4.249, 4.310, 4.545, 4.559, 4.666,
        4.789, 4.553, 5.211, 5.330, 5.520, 4.775, 4.439, 4.947, 5.131, 5.018,
        4.652, 4.552, 4.827, 4.700, 4.725, 4.918, 4.822, 4.867, 5.033, 4.942],
       device='cuda:0')

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,0].argsort().resize(50,1)].squeeze(),skip_special_tokens=True)

[' before my wife found out I was a man, I was the',
 ' before my wife found out I was a man, I had the',
 ' before my wife found out I was a man, I thought that',
 ' before my wife found out I was a man, I did not',
 ' before my wife found out I was a man, I had no',
 ' before my wife found out I was a man, I was not',
 ' before my wife found out I was a man, I thought the',
 ' before my wife found out I was a man, I had what',
 ' before my wife found out I was a man, I did the',
 ' before my wife found out I was a man, I had thought',
 ' before my wife found out I was a man, I knew that',
 ' before my wife found out I was a man, I was what',
 ' before my wife found out I was a man, I knew the',
 ' before my wife found out I was a man, I had not',
 ' before my wife found out I was a man, I did what',
 ' before my wife found out I was a man, I did everything',
 ' before my wife found out I was a man, I knew what',
 ' before my wife found out I was a man, I was no',
 ' before my wife fo

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,1].argsort().resize(50,1)].squeeze(),skip_special_tokens=True)

[' before my wife found out I was a man, I knew everything',
 ' before my wife found out I was a man, I had everything',
 ' before my wife found out I was a man, I knew the',
 ' before my wife found out I was a man, I had wondered',
 ' before my wife found out I was a man, I knew that',
 ' before my wife found out I was a man, I had the',
 ' before my wife found out I was a man, I knew what',
 ' before my wife found out I was a man, I had thought',
 ' before my wife found out I was a man, I had not',
 ' before my wife found out I was a man, I had that',
 ' before my wife found out I was a man, I thought that',
 ' before my wife found out I was a man, I did thought',
 ' before my wife found out I was a man, I had every',
 ' before my wife found out I was a man, I was that',
 ' before my wife found out I was a man, I did wondered',
 ' before my wife found out I was a man, I thought the',
 ' before my wife found out I was a man, I thought everything',
 ' before my wife found out I was a m

In [ ]:
torch.cat([logging_loss[:,0].argsort().resize(50,1),logging_loss[:,1].argsort().resize(50,1),curr_loss.argsort().resize(50,1)],dim=-1)

tensor([[25, 19, 25],
        [26, 16, 26],
        [12, 29, 12],
        [23,  6, 23],
        [36, 14, 36],
        [20, 26, 20],
        [27, 44, 27],
        [41, 31, 41],
        [28, 21, 31],
        [31, 11, 28],
        [14, 12, 14],
        [40, 33, 40],
        [29, 46, 29],
        [21, 10, 21],
        [43,  8, 43],
        [18, 27, 18],
        [44, 17, 44],
        [35, 13, 10],
        [10, 28,  6],
        [ 6, 49, 35],
        [30, 41, 30],
        [11,  1, 11],
        [17, 23, 16],
        [42,  9, 17],
        [16,  4, 46],
        [46,  5, 42],
        [47, 34, 47],
        [45, 36, 45],
        [13, 25, 13],
        [37, 47, 19],
        [19, 18, 49],
        [49,  3, 37],
        [15, 48, 15],
        [39, 24, 39],
        [48,  7, 48],
        [22, 32, 22],
        [ 5,  2,  5],
        [ 3, 43,  3],
        [24, 30, 24],
        [38, 22, 38],
        [32, 39, 32],
        [ 0, 42,  0],
        [33, 15, 33],
        [ 1, 37,  1],
        [ 8, 40,  8],
        [3

In [ ]:
        
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print(f"final_hypotheses: {final_hypotheses}")
print(f"final_hypotheses_losses: {final_hypotheses_losses}")

curr_loss: tensor([5.260, 5.338, 5.570, 5.089, 5.829, 5.084, 4.759, 5.611, 5.417, 5.680,
        4.753, 4.792, 4.394, 4.928, 4.603, 4.987, 4.807, 4.808, 4.705, 4.931,
        4.512, 4.675, 5.038, 4.435, 5.093, 4.249, 4.310, 4.545, 4.559, 4.666,
        4.789, 4.553, 5.211, 5.330, 5.520, 4.775, 4.439, 4.947, 5.131, 5.018,
        4.652, 4.552, 4.827, 4.700, 4.725, 4.918, 4.822, 4.867, 5.033, 4.942],
       device='cuda:0')
curr_loss: (tensor([5.260, 5.338, 5.570, 5.089, 5.829, 5.084, 4.759, 5.611, 5.417, 5.680,
        4.753, 4.792, 4.394, 4.928, 4.603, 4.987, 4.807, 4.808, 4.705, 4.931,
        4.512, 4.675, 5.038, 4.435, 5.093, 4.249, 4.310, 4.545, 4.559, 4.666,
        4.789, 4.553, 5.211, 5.330, 5.520, 4.775, 4.439, 4.947, 5.131, 5.018,
        4.652, 4.552, 4.827, 4.700, 4.725, 4.918, 4.822, 4.867, 5.033, 4.942],
       device='cuda:0'),)
top_beams: [tensor([25, 26, 12, 23, 36], device='cuda:0')]
tmp_hypotheses: (tensor([[    0,   137,   127,  1141,   303,    66,    38,    21,    1

In [ ]:
mlm_tokenizer.batch_decode(hypotheses[0])

['<s> before my wife found out I was a man, I was the<mask>',
 '<s> before my wife found out I was a man, I had the<mask>',
 '<s> before my wife found out I was a man, I thought that<mask>',
 '<s> before my wife found out I was a man, I did not<mask>',
 '<s> before my wife found out I was a man, I had no<mask>']

In [ ]:
mlm_tokenizer.batch_decode(final_hypotheses[0])

['<s> before my wife found out I was a man, I was',
 '<s> before my wife found out I was a man, I had',
 '<s> before my wife found out I was a man, I thought',
 '<s> before my wife found out I was a man, I did',
 '<s> before my wife found out I was a man, I knew',
 '<s> before my wife found out I was a man, I was the',
 '<s> before my wife found out I was a man, I had the',
 '<s> before my wife found out I was a man, I thought that',
 '<s> before my wife found out I was a man, I did not',
 '<s> before my wife found out I was a man, I had no']

In [ ]:
# 다음 step 
curr_edit_index = 14       
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
print(f"batch_ids_to_edit: {batch_ids_to_edit}") # [0]
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
new_func_candidates = new_func_candidates.to(config['device'])
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"len(tmp_hypotheses): {len(tmp_hypotheses)}")
print(f"tmp_hypotheses: {tmp_hypotheses}")

logging_loss = torch.zeros((len(tmp_hypotheses), len(lossfns))).to(config['device'])
curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
tmp_hypotheses_dec = mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True)
data_loader = DataLoader(CustomDataset(tmp_hypotheses_dec),batch_size=batch_size)
for lossid, lossname in enumerate(config["losses"]):
    lossvalues=[]
    with torch.no_grad():
        for batch in data_loader:
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, batch,
                label_id=config['target_label_ids'][lossid],
            )
            lossvalues.append(lossvalue)
            torch.cuda.empty_cache()
    lossvalue = torch.cat(lossvalues,dim=0)
    curr_loss += loss_weights[lossid] * lossvalue
    logging_loss[:,lossid] = lossvalue


batch_ids_to_edit: [0]
num_initial_tmp_hypotheses: [50]
len(tmp_hypotheses): 50
tmp_hypotheses: tensor([[   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
           21,    5,  117],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
           56,    5,  117],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
          802,   14,  117],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
          222,   45,  117],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
           56,  117,  117],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
           21,    5,   25],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
           56,    5,   25],
        [   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313,    6,   38,
          802,   14,   25],
        [   0,  137,  127, 1141,

In [ ]:
logging_loss

tensor([[   51.960,     0.071],
        [   52.497,     0.053],
        [   50.225,     0.037],
        [   54.506,     0.037],
        [   53.650,     0.055],
        [   53.107,     0.035],
        [   54.776,     0.026],
        [   49.956,     0.024],
        [   53.097,     0.033],
        [   54.611,     0.037],
        [   53.868,     0.043],
        [   55.737,     0.018],
        [   50.493,     0.021],
        [   54.763,     0.026],
        [   55.333,     0.028],
        [   53.332,     0.076],
        [   52.710,     0.037],
        [   49.346,     0.037],
        [   53.138,     0.025],
        [   54.370,     0.036],
        [   45.053,     0.043],
        [   49.923,     0.031],
        [   49.638,     0.025],
        [   51.390,     0.028],
        [   54.991,     0.043],
        [   50.633,     0.016],
        [   53.024,     0.026],
        [   53.404,     0.020],
        [   55.870,     0.039],
        [   49.850,     0.034],
        [   53.094,     0.035],
        

In [ ]:
curr_loss

tensor([5.267, 5.303, 5.060, 5.488, 5.420, 5.346, 5.503, 5.020, 5.343, 5.498,
        5.430, 5.591, 5.070, 5.503, 5.561, 5.409, 5.308, 4.971, 5.339, 5.473,
        4.548, 5.023, 4.989, 5.167, 5.542, 5.080, 5.329, 5.360, 5.626, 5.019,
        5.344, 4.822, 5.462, 5.457, 5.177, 5.706, 5.686, 5.680, 5.858, 5.717,
        4.628, 5.215, 5.036, 5.529, 5.230, 5.289, 5.382, 5.254, 4.834, 5.567],
       device='cuda:0')

In [ ]:
torch.cat([logging_loss[:,0].argsort().resize(50,1),logging_loss[:,1].argsort().resize(50,1),curr_loss.argsort().resize(50,1)],dim=-1)

tensor([[20, 40, 20],
        [40, 25, 40],
        [48, 11, 31],
        [31, 31, 48],
        [17, 27, 17],
        [22, 12, 22],
        [29, 36, 29],
        [21,  7,  7],
        [ 7, 22, 21],
        [42, 32, 42],
        [ 2, 18,  2],
        [12,  6, 12],
        [25, 37, 25],
        [23, 13, 23],
        [44, 26, 34],
        [34, 34, 41],
        [41, 14, 44],
        [ 0, 23, 47],
        [47, 41,  0],
        [45, 47, 45],
        [ 1, 33,  1],
        [16, 21, 16],
        [26, 38, 26],
        [30,  8, 18],
        [ 8, 39,  8],
        [ 5, 29, 30],
        [18, 30,  5],
        [15, 46, 27],
        [27,  5, 46],
        [46, 19, 15],
        [ 4, 16,  4],
        [10, 17, 10],
        [33,  9, 33],
        [32, 42, 32],
        [19,  2, 19],
        [43,  3,  3],
        [ 3, 35,  9],
        [ 9, 28, 13],
        [49, 24,  6],
        [13, 20, 43],
        [ 6, 10, 24],
        [24, 45, 14],
        [14,  1, 49],
        [11,  4, 11],
        [28,  0, 28],
        [3

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,0].argsort().resize(50,1)].squeeze(),skip_special_tokens=True)

[' before my wife found out I was a man, I was the only',
 ' before my wife found out I was a man, I was the man',
 ' before my wife found out I was a man, I did not like',
 ' before my wife found out I was a man, I had the thought',
 ' before my wife found out I was a man, I thought that what',
 ' before my wife found out I was a man, I thought that only',
 ' before my wife found out I was a man, I had no way',
 ' before my wife found out I was a man, I had the only',
 ' before my wife found out I was a man, I thought that as',
 ' before my wife found out I was a man, I thought that man',
 ' before my wife found out I was a man, I thought that no',
 ' before my wife found out I was a man, I thought that that',
 ' before my wife found out I was a man, I was the way',
 ' before my wife found out I was a man, I did not only',
 ' before my wife found out I was a man, I had no man',
 ' before my wife found out I was a man, I had no thought',
 ' before my wife found out I was a man, I had t

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[logging_loss[:,1].argsort().resize(50,1)].squeeze(),skip_special_tokens=True)

[' before my wife found out I was a man, I was the man',
 ' before my wife found out I was a man, I was the way',
 ' before my wife found out I was a man, I had the that',
 ' before my wife found out I was a man, I had the thought',
 ' before my wife found out I was a man, I thought that way',
 ' before my wife found out I was a man, I thought that that',
 ' before my wife found out I was a man, I had the than',
 ' before my wife found out I was a man, I thought that as',
 ' before my wife found out I was a man, I thought that only',
 ' before my wife found out I was a man, I thought that thought',
 ' before my wife found out I was a man, I did not what',
 ' before my wife found out I was a man, I had the as',
 ' before my wife found out I was a man, I thought that than',
 ' before my wife found out I was a man, I did not that',
 ' before my wife found out I was a man, I had the way',
 ' before my wife found out I was a man, I had no thought',
 ' before my wife found out I was a man, I

In [ ]:
mlm_tokenizer.batch_decode(tmp_hypotheses[curr_loss.argsort().resize(50,1)].squeeze(),skip_special_tokens=True)

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/torch/_tensor.py:889: UserWarning: non-inplace resize is deprecated
  warnings.warn("non-inplace resize is deprecated")


[' before my wife found out I was a man, I was the only',
 ' before my wife found out I was a man, I was the man',
 ' before my wife found out I was a man, I had the thought',
 ' before my wife found out I was a man, I did not like',
 ' before my wife found out I was a man, I thought that what',
 ' before my wife found out I was a man, I thought that only',
 ' before my wife found out I was a man, I had no way',
 ' before my wife found out I was a man, I thought that as',
 ' before my wife found out I was a man, I had the only',
 ' before my wife found out I was a man, I thought that man',
 ' before my wife found out I was a man, I thought that no',
 ' before my wife found out I was a man, I thought that that',
 ' before my wife found out I was a man, I was the way',
 ' before my wife found out I was a man, I did not only',
 ' before my wife found out I was a man, I had no thought',
 ' before my wife found out I was a man, I had the man',
 ' before my wife found out I was a man, I had 

In [ ]:

print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print(f"final_hypotheses: {final_hypotheses}")
print(f"final_hypotheses_losses: {final_hypotheses_losses}")


curr_loss: tensor([5.267, 5.303, 5.060, 5.488, 5.420, 5.346, 5.503, 5.020, 5.343, 5.498,
        5.430, 5.591, 5.070, 5.503, 5.561, 5.409, 5.308, 4.971, 5.339, 5.473,
        4.548, 5.023, 4.989, 5.167, 5.542, 5.080, 5.329, 5.360, 5.626, 5.019,
        5.344, 4.822, 5.462, 5.457, 5.177, 5.706, 5.686, 5.680, 5.858, 5.717,
        4.628, 5.215, 5.036, 5.529, 5.230, 5.289, 5.382, 5.254, 4.834, 5.567],
       device='cuda:0')
curr_loss: (tensor([5.267, 5.303, 5.060, 5.488, 5.420, 5.346, 5.503, 5.020, 5.343, 5.498,
        5.430, 5.591, 5.070, 5.503, 5.561, 5.409, 5.308, 4.971, 5.339, 5.473,
        4.548, 5.023, 4.989, 5.167, 5.542, 5.080, 5.329, 5.360, 5.626, 5.019,
        5.344, 4.822, 5.462, 5.457, 5.177, 5.706, 5.686, 5.680, 5.858, 5.717,
        4.628, 5.215, 5.036, 5.529, 5.230, 5.289, 5.382, 5.254, 4.834, 5.567],
       device='cuda:0'),)
top_beams: [tensor([20, 40, 31, 48, 17], device='cuda:0')]
tmp_hypotheses: (tensor([[   0,  137,  127, 1141,  303,   66,   38,   21,   10,  313, 

In [ ]:
mlm_tokenizer.batch_decode(hypotheses[0])

['<s> before my wife found out I was a man, I was the only',
 '<s> before my wife found out I was a man, I was the man',
 '<s> before my wife found out I was a man, I had the thought',
 '<s> before my wife found out I was a man, I did not like',
 '<s> before my wife found out I was a man, I thought that what']

In [ ]:
mlm_tokenizer.batch_decode(final_hypotheses[0])

['<s> before my wife found out I was a man, I was',
 '<s> before my wife found out I was a man, I had',
 '<s> before my wife found out I was a man, I thought',
 '<s> before my wife found out I was a man, I did',
 '<s> before my wife found out I was a man, I knew',
 '<s> before my wife found out I was a man, I was the',
 '<s> before my wife found out I was a man, I had the',
 '<s> before my wife found out I was a man, I thought that',
 '<s> before my wife found out I was a man, I did not',
 '<s> before my wife found out I was a man, I had no',
 '<s> before my wife found out I was a man, I was the only',
 '<s> before my wife found out I was a man, I was the man',
 '<s> before my wife found out I was a man, I had the thought',
 '<s> before my wife found out I was a man, I did not like',
 '<s> before my wife found out I was a man, I thought that what']

In [ ]:
## decoding 완료 & 함수에서 return 문
return_hypotheses = [mlm_tokenizer.batch_decode(x, skip_special_tokens=True) for x in final_hypotheses], final_hypotheses_losses

In [ ]:
## beam 함수 바깥에서 하는 final reranking for this span
hypotheses=list(queue) # deletion case
hypotheses.extend(return_hypotheses[0][0])

In [ ]:
hypotheses

[' before my wife found out I was a man, I',
 ' before my wife found out I was a man, I was',
 ' before my wife found out I was a man, I had',
 ' before my wife found out I was a man, I thought',
 ' before my wife found out I was a man, I did',
 ' before my wife found out I was a man, I knew',
 ' before my wife found out I was a man, I was the',
 ' before my wife found out I was a man, I had the',
 ' before my wife found out I was a man, I thought that',
 ' before my wife found out I was a man, I did not',
 ' before my wife found out I was a man, I had no',
 ' before my wife found out I was a man, I was the only',
 ' before my wife found out I was a man, I was the man',
 ' before my wife found out I was a man, I had the thought',
 ' before my wife found out I was a man, I did not like',
 ' before my wife found out I was a man, I thought that what']

In [ ]:
hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]

In [ ]:
# Scoring the hypotheses and select top beam hypotheses
curr_loss = torch.zeros(len(hypotheses_all)).to(config['device'])
data_loader = DataLoader(CustomDataset(hypotheses_all),batch_size=batch_size)
logging_loss = torch.zeros((len(hypotheses_all),len(lossfns))).to(config['device'])

for lossid, lossname in enumerate(config["losses"]):
    lossvalues=[]
    with torch.no_grad():
        for batch in data_loader:
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, batch,
                label_id=config['target_label_ids'][lossid],
            )
            lossvalues.append(lossvalue)
            torch.cuda.empty_cache()
    lossvalue = torch.cat(lossvalues,dim=0)
    curr_loss += config['loss_weights'][lossid] * lossvalue
    logging_loss[:, lossid] = lossvalue.clone()

torch.cuda.empty_cache()

In [ ]:
logging_loss

tensor([[147.203,   2.784],
        [150.385,   2.723],
        [147.504,   2.747],
        [148.891,   2.679],
        [151.666,   2.706],
        [152.609,   2.758],
        [154.191,   2.799],
        [155.248,   2.725],
        [150.477,   2.791],
        [154.100,   2.860],
        [147.822,   2.767],
        [147.352,   2.759],
        [156.486,   2.813],
        [153.878,   2.887],
        [153.277,   2.758],
        [145.937,   2.745],
        [144.829,   2.738],
        [140.819,   2.672],
        [147.932,   2.668],
        [140.957,   2.661],
        [156.149,   2.705],
        [156.844,   2.741],
        [157.380,   2.679],
        [154.247,   2.771],
        [158.197,   2.821],
        [150.534,   2.723],
        [150.335,   2.725],
        [161.738,   2.789],
        [157.274,   2.870],
        [159.209,   2.656],
        [148.623,   2.710],
        [148.122,   2.716],
        [151.139,   2.608],
        [144.194,   2.649],
        [149.190,   2.726],
        [153.062,   

In [ ]:
curr_loss

tensor([17.504, 17.761, 17.497, 17.568, 17.872, 18.019, 18.218, 18.250, 17.839,
        18.270, 17.549, 17.494, 18.461, 18.274, 18.086, 17.339, 17.221, 16.754,
        17.461, 16.757, 18.320, 18.425, 18.417, 18.196, 18.641, 17.776, 17.758,
        18.962, 18.597, 18.577, 17.573, 17.528, 17.722, 17.068, 17.645, 18.026,
        17.950, 18.202, 18.374, 18.273, 17.455, 17.350, 18.380, 17.671, 18.632,
        17.178, 17.183, 17.138, 17.102, 17.310, 18.331, 18.365, 18.078, 18.452,
        18.161, 18.057, 18.421, 17.879, 17.809, 17.896, 17.569, 17.430, 17.534,
        17.841, 17.779, 18.401, 18.584, 18.541, 18.419, 18.641, 17.899, 17.938,
        17.623, 18.031, 18.147, 17.713, 17.432, 17.651, 17.371, 17.810],
       device='cuda:0')

In [ ]:
[hypotheses_all[i] for i in logging_loss[:,0].argsort()]

[' before my wife found out I was a man, I thought one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought that one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man, I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought one could be a woman and I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a woman, I thought I was simply some k

In [ ]:
[hypotheses_all[i] for i in logging_loss[:,1].argsort()]

[' before my wife found out I was a man, I one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I had one could be a boy. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I knew one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I had one could be a wife. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I had one could be a girl. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man. I thought I was simply some kind of crazy old fool. My m

In [ ]:
[hypotheses_all[i] for i in curr_loss.argsort()]

[' before my wife found out I was a man, I thought one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought that one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man, I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a woman. I thought I was simply some kind of crazy 

In [ ]:
top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices
queue = [hypotheses_all[ix] for ix in top_beams]

In [ ]:
queue

[' before my wife found out I was a man, I thought one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I thought that one could be a woman. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man. I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local',
 ' before my wife found out I was a man, I one could be a man, I thought I was simply some kind of crazy old fool. My mom had no sympathy and I found myself in the local']